# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [2]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one: line item from an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [3]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [4]:
# TODO
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined[joined['_merge'] == 'left_only']

print('Unmatched vendor:', unmatched['vendor_id'].iloc[0])
print('Units attached to unmatched vendor:', unmatched['qty'].sum())

joined

Unmatched vendor: V-07
Units attached to unmatched vendor: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [5]:
# TODO
vendor_totals_data = joined.copy()
vendor_totals_data['name'] = vendor_totals_data['name'].fillna('Unknown / unmatched vendor')

vendor_totals = vendor_totals_data.groupby('name')['qty'].sum()
vendor_totals

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown / unmatched vendor,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [6]:
# TODO
zone_totals = joined.groupby('zone')['qty'].sum()

print('Zone A sold the most items.')
print('Quantity included in zone totals:', zone_totals.sum())
print('Quantity missing from zone totals:', joined[joined['zone'].isna()]['qty'].sum())
print('The missing 3 units belong to V-07, which had no matching vendor record and therefore no zone.')

zone_totals

Zone A sold the most items.
Quantity included in zone totals: 4
Quantity missing from zone totals: 3
The missing 3 units belong to V-07, which had no matching vendor record and therefore no zone.


,qty
zone,
A,4


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [7]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_with_lines = [order for order in ragged_json if 'lines' in order]

ragged_lines = pd.json_normalize(
    ragged_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [8]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) one row per line item is useful here because eacvh item and its quantity can be analyzed directly. For instance, it becomes easier to total the quanity sold by vendor or zone with a groupby. if there were one row per order, the individual items would still be nested inside the lines list, so i would have to unpack those lists before doing those calculations

b) A missing qunatity means the quantity is unknown or was not provided, while a quantity with zero means we know that zero units were recorded. If i replace a missing quantity with zero, the later calculations would treat an unknown value as a real observation. This could hide a data-quality problem and make totals, averages, or counts a bit misleading and off.